In [2]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('sqlite:///../ecommerce.db')

check = pd.read_sql("SELECT COUNT(*) AS total_rows FROM orders;", engine)
print(check)

   total_rows
0       99441


In [3]:
query = """
SELECT
    c.customer_unique_id,
    MAX(o.order_purchase_timestamp) AS last_purchase_date,
    COUNT(DISTINCT o.order_id) AS frequency,
    ROUND(SUM(oi.price), 2) AS monetary
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_unique_id;
"""

rfm = pd.read_sql(query, engine)
print(rfm.shape)
print(rfm['frequency'].value_counts().head())
rfm.head()

(93358, 4)
frequency
1    90557
2     2573
3      181
4       28
5        9
Name: count, dtype: int64


,customer_unique_id,last_purchase_date,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27.000000,1,129.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27.000000,1,18.90
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03.000000,1,69.00
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41.000000,1,25.99
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42.000000,1,180.00


In [5]:
print("Total customers:", len(rfm))
print()
print(rfm['frequency'].value_counts())

Total customers: 93358

frequency
1     90557
2      2573
3       181
4        28
5         9
6         5
7         3
9         1
15        1
Name: count, dtype: int64


In [6]:
rfm['last_purchase_date'] = pd.to_datetime(rfm['last_purchase_date'])

reference_date = rfm['last_purchase_date'].max()

rfm['recency'] = (reference_date - rfm['last_purchase_date']).dt.days

rfm[['recency', 'frequency', 'monetary']].describe()

,recency,frequency,monetary
count,93358.000000,93358.000000,93358.000000
mean,236.941773,1.033420,141.621480
std,152.591453,0.209097,215.694014
min,0.000000,1.000000,0.850000
25%,113.000000,1.000000,47.650000
50%,218.000000,1.000000,89.730000
75%,345.000000,1.000000,154.737500
max,713.000000,15.000000,13440.000000


In [7]:
rfm['r_score'] = pd.qcut(rfm['recency'], 4, labels=[4, 3, 2, 1]).astype(int)

rfm['m_score'] = pd.qcut(rfm['monetary'], 4, labels=[1, 2, 3, 4]).astype(int)

rfm['f_score'] = rfm['frequency'].apply(lambda x: 1 if x == 1 else (2 if x == 2 else 3))

print(rfm['r_score'].value_counts().sort_index())
print(rfm['f_score'].value_counts().sort_index())
print(rfm['m_score'].value_counts().sort_index())

r_score
1    23311
2    23200
3    23451
4    23396
Name: count, dtype: int64
f_score
1    90557
2     2573
3      228
Name: count, dtype: int64
m_score
1    23341
2    23338
3    23339
4    23340
Name: count, dtype: int64


In [8]:
def segment_customer(row):
    if row['f_score'] >= 2 and row['r_score'] >= 3:
        return 'Champions'
    elif row['f_score'] >= 2:
        return 'Loyal Customers'
    elif row['r_score'] >= 3:
        return 'Recent Customers'
    elif row['m_score'] >= 3:
        return 'At Risk'
    else:
        return 'Lost/Low Value'

rfm['segment'] = rfm.apply(segment_customer, axis=1)

print(rfm['segment'].value_counts())

segment
Recent Customers    45298
Lost/Low Value      23283
At Risk             21976
Champions            1549
Loyal Customers      1252
Name: count, dtype: int64


In [9]:
rfm.to_sql('customer_rfm_unique', engine, if_exists='replace', index=False)
rfm.to_csv('../data/cleaned/customer_rfm_unique.csv', index=False)

print("Saved", len(rfm), "customers")

Saved 93358 customers
